In [1]:
# ========== 第 1 周练习 1：从邮件里抽日历事件（返回 JSON）==========
# 理念：练习 system / user prompt + messages 列表 + 一次 Chat Completions 调用
# 和本课关系：Day1 的「写好提示词 → 组装 messages → 调模型 → 读 content」完整闭环

# 导入标准库 os：读环境变量（本格未直接用，但常与密钥配置一起出现）
import os
# 从 dotenv 导入 load_dotenv：可把 .env 密钥读进环境（本格未调用，保留原导入）
from dotenv import load_dotenv
# 从 IPython.display 导入 Markdown / display：需要时在笔记本里漂亮展示（本格用 print）
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI

# 创建默认 OpenAI 客户端（密钥通常来自 OPENAI_API_KEY）
openai = OpenAI()

# ---------- 步骤 1：写好 system / user 提示词，并准备待分析的邮件正文 ----------

# system：角色设定——专业助理，从复杂邮件抽日期并放进日历；只要合法 JSON（字符串保持英文）
system_prompt = """
            You are a professional assistant who is very thorough in analyzing complex emails, 
            extract the dates from it and insert them in the calendar. Return ONLY valid JSON.
            """
# user：任务说明——摘要邮件、抽出日历相关事件并列出（后面会拼上 email 正文）
user_prompt = """
    Here is the text you need to analyze.
    Summarize my email, extracts any calendar specific events from it and lists them for me.
"""

# 待分析的英文邮件样例：含董事会、假期、提案截止等相对时间表述
email = """Let’s reconnect a couple of days after the quarterly board meeting, which is slated for the 25th. 

I’ll be on leave the week following Holi, returning the Monday right after. 

Please circulate the revised proposal by mid-next week so we have buffer before month-end closing."""

# ---------- 步骤 2：组装 messages 列表（Chat Completions 标准格式）----------

messages = [
    # system 消息：定行为边界（只返回 JSON）
    {"role":"system","content":system_prompt},
    # user 消息：说明 + 邮件原文拼接
    {"role":"user","content":user_prompt + email}
] # fill this in

# ---------- 步骤 3：调用 OpenAI Chat Completions ----------

# model 用 gpt-4.1-nano（字符串保持原样）；把 messages 整表传给 API
response = openai.chat.completions.create(model="gpt-4.1-nano",messages = messages)
# 取出第一条 choice 的助手回复正文（表达式本身会显示在 notebook 输出里）
response.choices[0].message.content

# ---------- 步骤 4：打印结果，方便复制 / 核对是否为合法 JSON ----------
print(response.choices[0].message.content)


{
  "events": [
    {
      "event": "Quarterly board meeting",
      "date": "2023-10-25"
    },
    {
      "event": "Reconnect",
      "date": "2023-10-27"
    },
    {
      "event": "Revised proposal circulation deadline",
      "date": "2023-10-31"
    },
    {
      "event": "Holi",
      "date": "2024-03-25"
    },
    {
      "event": "Leave start (following Holi)",
      "date": "2024-03-26"
    },
    {
      "event": "Return from leave (Monday after Holi)",
      "date": "2024-03-29"
    }
  ]
}
